# Fashion-MNIST: Controlled Hyperparameter Optimization

**CO3133 — Deep Learning · Assignment 1 · Group G-M10**

This notebook executes a reproducible optimization study for Linear, MLP, CNN, GRU and Transformer classifiers. The protocol comprises 36 discovery runs, 10 paired confirmation runs and 15 final runs, with 60 epochs per completed run.

The study uses training and validation data only. Hyperparameter selection is exploratory because the original official-test results were already available when this extension was designed. New validation scores are therefore not an independent test evaluation.

Execution is bound to a verified source archive, a fixed experiment manifest and persistent artifacts on Google Drive. The configured budget is 604,800 seconds of cumulative controller activity; the study ends as soon as all 61 runs are complete.


## 1. Environment and source verification

The following cell validates NVIDIA A100 availability, mounts persistent storage and verifies the source archive and every manifest entry before extraction. It records the CUDA software environment and restores saved non-CUDA dependency versions when resuming an existing run.

The source fingerprint, output directory and environment checks prevent incompatible files or software versions from being combined within one study.


In [ ]:
import hashlib, json, os, shutil, stat, subprocess, sys, tempfile, zipfile
from pathlib import Path, PurePosixPath
from google.colab import drive, files

os.chdir("/content")
PREFLIGHT_OK = PLAN_OK = SMOKE_OK = False
RUN_REQUESTED = TRAINING_RETURNED_OK = False
SOURCE_ZIP_SHA256 = "1b40cb3e9a6abb86b6e7f00ba15800d1781f0dae4ee8f985dd470ba27d34740c"
SOURCE_NAME = "CO3133_A1_Optimization_v3_1_7Days_Source.zip"
REPO = Path("/content/CO3133-A1-opt3-v3-1-7days")
CACHED_SOURCE = Path("/content/drive/MyDrive/G-M10_A1_Optimization_v3_1_7days/source") / SOURCE_NAME
LOCAL_SOURCE = Path("/content") / SOURCE_NAME
DEFAULT_OUT = Path("/content/drive/MyDrive/G-M10_A1_Optimization_v3_1_7days/runs/g_m10_a1_opt3_v3_1_7days_run01")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def checked(args, cwd=None):
    return subprocess.run([str(arg) for arg in args], cwd=cwd, check=True, text=True)

assert len(SOURCE_ZIP_SHA256) == 64 and all(c in "0123456789abcdef" for c in SOURCE_ZIP_SHA256), (
    "Notebook chưa được đóng gói với hash source. Dùng đúng notebook được bàn giao.")
checked(["nvidia-smi"])
gpu_names = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip().splitlines()
assert gpu_names and all("A100" in name for name in gpu_names), (
    "Cần NVIDIA A100. Chọn A100 trong Runtime → Change runtime type rồi chạy lại.")

drive.mount("/content/drive")
if CACHED_SOURCE.is_file() and sha256_file(CACHED_SOURCE) == SOURCE_ZIP_SHA256:
    shutil.copyfile(CACHED_SOURCE, LOCAL_SOURCE)
    print("Đã lấy đúng source từ Drive; không cần upload lại.")
else:
    if CACHED_SOURCE.exists():
        print("Source lưu trên Drive khác bản này. Upload đúng ZIP được giao cùng notebook.")
    uploaded = files.upload()
    names = [name for name in uploaded if Path(name).name == SOURCE_NAME]
    assert len(names) == 1, "Chọn đúng một tệp: " + SOURCE_NAME
    LOCAL_SOURCE.write_bytes(uploaded[names[0]])
    del uploaded
assert sha256_file(LOCAL_SOURCE) == SOURCE_ZIP_SHA256, (
    "Hash source ZIP không khớp. Không tiếp tục; gửi thông báo này để kiểm tra tệp.")

with zipfile.ZipFile(LOCAL_SOURCE) as archive:
    infos = archive.infolist()
    assert len(infos) <= 2000 and sum(info.file_size for info in infos) <= 256 * 1024**2, (
        "Source ZIP có kích thước bất thường.")
    archive_files = {}
    for info in infos:
        name = info.filename
        path = PurePosixPath(name)
        assert not path.is_absolute() and ".." not in path.parts and "\\" not in name, (
            "Đường dẫn ZIP không hợp lệ: " + name)
        if info.is_dir():
            continue
        assert name == str(path) and name not in archive_files, "Tên tệp ZIP không hợp lệ: " + name
        assert not stat.S_ISLNK(info.external_attr >> 16), "Không chấp nhận symlink trong source."
        archive_files[name] = info
    assert "SOURCE_MANIFEST.json" in archive_files, "Thiếu SOURCE_MANIFEST.json."
    manifest_bytes = archive.read("SOURCE_MANIFEST.json")
    source_manifest = json.loads(manifest_bytes)
    payload_hashes = source_manifest.get("files")
    assert isinstance(payload_hashes, dict) and payload_hashes, "Manifest source không hợp lệ."
    assert set(archive_files) == set(payload_hashes) | {"SOURCE_MANIFEST.json"}, (
        "Danh sách tệp trong ZIP khác manifest.")
    for name, expected in payload_hashes.items():
        assert isinstance(expected, str) and len(expected) == 64, "Hash tệp không hợp lệ: " + name
        assert hashlib.sha256(archive.read(name)).hexdigest() == expected, "Sai hash tệp: " + name
    if not REPO.exists():
        with tempfile.TemporaryDirectory(prefix="a1-opt3-", dir="/content") as temporary:
            staged = Path(temporary) / "source"
            staged.mkdir()
            for name in archive_files:
                target = staged.joinpath(*PurePosixPath(name).parts)
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(name))
            os.replace(staged, REPO)

assert REPO.is_dir(), "Đường dẫn source hiện tại không phải thư mục."
assert (REPO / "SOURCE_MANIFEST.json").read_bytes() == manifest_bytes, (
    "Source local khác bản bàn giao. Giữ output Drive, mở runtime sạch rồi dùng đúng source này.")
for name, expected in payload_hashes.items():
    assert (REPO / name).is_file() and sha256_file(REPO / name) == expected, (
        "Source local đã đổi hoặc thiếu: " + name)
unexpected = [str(path.relative_to(REPO)) for path in REPO.rglob("*")
              if path.is_file() and "__pycache__" not in path.parts and path.suffix != ".pyc"
              and path.relative_to(REPO).as_posix() not in archive_files]
assert not unexpected, "Source local có tệp ngoài manifest: " + ", ".join(unexpected[:5])

CACHED_SOURCE.parent.mkdir(parents=True, exist_ok=True)
if not CACHED_SOURCE.is_file() or sha256_file(CACHED_SOURCE) != SOURCE_ZIP_SHA256:
    temporary_cache = CACHED_SOURCE.with_suffix(".zip.partial")
    shutil.copyfile(LOCAL_SOURCE, temporary_cache)
    assert sha256_file(temporary_cache) == SOURCE_ZIP_SHA256, "Ghi source lên Drive chưa khớp."
    os.replace(temporary_cache, CACHED_SOURCE)
assert sha256_file(CACHED_SOURCE) == SOURCE_ZIP_SHA256, "Không đọc lại được đúng source từ Drive."

os.chdir(REPO)
for required in ("tools/run_a1_optimization.py", "configs/a1_optimization_v3.json",
                 "requirements-a1-optimization.txt"):
    assert (REPO / required).is_file(), "Thiếu tệp bắt buộc: " + required
saved_environment = DEFAULT_OUT / "run_environment.json"
saved_versions = {}
if saved_environment.is_file():
    saved_versions = json.loads(saved_environment.read_text()).get("versions", {})
    package_names = ("numpy", "scikit-learn", "optuna", "matplotlib")
    assert all(isinstance(saved_versions.get(name), str) for name in package_names), (
        "Biên nhận môi trường cũ không đủ phiên bản; dừng để kiểm tra trước khi resume.")
    checked([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
             *[name + "==" + saved_versions[name] for name in package_names]])
    print("Đã khôi phục phiên bản các thư viện ngoài CUDA từ run trên Drive.")
else:
    checked([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
             "-r", REPO / "requirements-a1-optimization.txt"])
from importlib.metadata import version as package_version
import torch, torchvision
assert torch.cuda.is_available(), "CUDA chưa hoạt động; kiểm tra runtime A100."
assert "A100" in torch.cuda.get_device_name(0), "GPU thực tế khác A100."
if saved_versions:
    assert package_version("torch") == saved_versions.get("torch"), (
        "Bản torch CUDA của Colab khác run đã lưu; không ép resume hoặc tự cài lại CUDA.")
    assert package_version("torchvision") == saved_versions.get("torchvision"), (
        "Bản torchvision khác run đã lưu; dừng và gửi thông báo để kiểm tra.")
print("GPU thực tế:", torch.cuda.get_device_name(0))
print("Python:", sys.version.split()[0], "torch:", torch.__version__,
      "torchvision:", torchvision.__version__, "CUDA:", torch.version.cuda)
print("Source ZIP SHA-256:", SOURCE_ZIP_SHA256)
print("Source:", REPO)
PREFLIGHT_OK = True
print("BƯỚC 1 OK")


## 2. Experiment manifest and data partition

This cell establishes the output location and invokes protocol preparation. Preparation verifies the dataset fingerprint, preserves the original 54,000/6,000 train–validation split and creates a fixed stratified 48,600/5,400 inner split for discovery and confirmation.

Candidate configurations, seeds, source identity and split hashes are stored in an immutable study manifest. The configured time budget is read directly from the verified protocol; this stage does not perform full training.


In [ ]:
assert PREFLIGHT_OK, "Chạy thành công bước 1 trước."
PLAN_OK = False
OUT = DEFAULT_OUT
DATA = Path("/content/fashion_mnist")
PROTOCOL = Path("configs/a1_optimization_v3.json")

def run_optimization(stage):
    return checked([sys.executable, "-u", "tools/run_a1_optimization.py", "--stage", stage,
                    "--protocol", PROTOCOL, "--out", OUT, "--data", DATA,
                    "--device", "cuda"], cwd=REPO)

print("Kết quả sẽ lưu:", OUT)
print("36 discovery + 10 confirmation + 15 final = tối đa 61 lượt; 60 epoch/lượt.")
budget_seconds = json.loads((REPO / PROTOCOL).read_text(encoding="utf-8"))["time_budget_seconds"]
assert budget_seconds == 604800, "Source này phải có ngân sách 7 ngày (604800 giây)."
print(f"Chỉ train/validation. Không official test. Ngân sách tối đa: {budget_seconds / 86400:g} ngày ({budget_seconds} giây).")
print("Hoàn tất 61 lượt sẽ kết thúc ngay; không chờ đủ 7 ngày.")
run_optimization("plan")
PLAN_OK = True
print("BƯỚC 2 OK")


## 3. Pipeline smoke validation

The smoke stage checks finite forward and backward computations and one optimizer step on eight training images for each of the 36 candidate configurations. It also verifies a checkpoint round trip for the anchor configuration of each model family.

These checks validate software behavior and checkpoint compatibility. They are not model-performance measurements and do not contribute results to the study.


In [ ]:
assert PLAN_OK, "Chạy thành công bước 2 trước."
SMOKE_OK = False
run_optimization("smoke")
SMOKE_OK = True
print("BƯỚC 3 OK — training đã bật ở bước 4; Run all sẽ tiếp tục chạy.")


## 4. Budgeted training and model selection

The controller executes discovery, paired confirmation and final training in sequence. All runs optimize cross-entropy; the primary checkpoint maximizes validation macro-F1, with lower cross-entropy and then the earlier epoch resolving exact ties. The lowest-cross-entropy checkpoint is retained separately.

Discovery and confirmation use the inner split. After recipe selection, each model family is trained from scratch on the original 54,000 training images with seeds 42, 43 and 44; the original 6,000 validation images select the final checkpoints.

Completed runs are verified and skipped during resume. Interrupted runs continue from their committed checkpoints. Time forecasts guide resource reservation, while the accumulated controller clock retains the configured budget across sessions. Official-test evaluation is outside this study.


In [ ]:
RUN_OPTIMIZATION = True

assert SMOKE_OK, "Smoke ở bước 3 chưa thành công."
RUN_REQUESTED = RUN_OPTIMIZATION
TRAINING_RETURNED_OK = False
if RUN_OPTIMIZATION:
    run_optimization("run")
    TRAINING_RETURNED_OK = True
    print("Lượt chạy đã trả về. Xem bước 5 để biết COMPLETE hay còn thiếu.")
else:
    print("SKIPPED: chưa bật full training. Đổi RUN_OPTIMIZATION = True ở bước 4 rồi chạy.")


## 5. Artifact verification and export

This cell validates the current study status and exports a ZIP archive containing the available manifests, source snapshot, environment records, checkpoints, metrics and analysis artifacts. The export receipt records the archive fingerprint and the number of completed runs.

A complete study requires a verified `COMPLETE` status and 61 completed runs. A partial archive preserves available evidence without representing an unfinished study as complete. The exported ZIP remains on Google Drive and is also offered for local download.


In [ ]:
DOWNLOAD_RESULTS = True

assert PLAN_OK, "Cần hoàn thành bước 2 để có study hợp lệ."
run_optimization("status")
run_optimization("export")
RESULT_ZIP = OUT.parent / (OUT.name + "_results.zip")
EXPORT_RECEIPT = OUT / "export_receipt.json"
assert RESULT_ZIP.is_file(), "Controller chưa tạo ZIP kết quả."
assert EXPORT_RECEIPT.is_file(), "Thiếu export_receipt.json; giữ log để kiểm tra."
print("ZIP kết quả trên Drive:", RESULT_ZIP)
print("Biên nhận export:", EXPORT_RECEIPT)
print("SHA-256 ZIP:", sha256_file(RESULT_ZIP))
receipt = json.loads(EXPORT_RECEIPT.read_text(encoding="utf-8"))
completed = receipt.get("completed_tasks", 0)
if receipt.get("status") == "COMPLETE" and completed == 61:
    print("HOÀN TẤT: 61/61 lượt. Gửi ZIP để kiểm tra số liệu trước khi đưa vào báo cáo.")
else:
    print(f"ZIP MỘT PHẦN: {completed}/61 lượt. Nghiên cứu chưa hoàn tất.")
    print("Chỉ có ZIP chưa có nghĩa là có kết quả tối ưu đầy đủ; gửi trạng thái để kiểm tra.")
if not RUN_REQUESTED:
    print("Lưu ý: bước 4 đang False; ZIP có thể chỉ chứa plan/smoke/trạng thái.")
if DOWNLOAD_RESULTS:
    files.download(str(RESULT_ZIP))


## Recorded results

The completed student-run study contains 61 tasks, each with 60 epochs. The selected final recipes use seeds 42, 43 and 44. Values below are validation mean ± sample SD on the same 6,000 checkpoint-selection images.

| Model | Recipe | Accuracy (%) | Macro-F1 |
|---|---|---:|---:|
| Linear | C1 | 86.856 ± 0.010 | 0.86851 ± 0.00024 |
| MLP | C7 | 91.439 ± 0.167 | 0.91448 ± 0.00153 |
| CNN | C5 | 93.278 ± 0.059 | 0.93308 ± 0.00027 |
| GRU | C1 | 91.400 ± 0.328 | 0.91367 ± 0.00376 |
| Transformer | C1 | 91.433 ± 0.120 | 0.91431 ± 0.00096 |

Accuracy SD is in percentage points. These are exploratory selected-validation scores; original frozen official-test results remain separate.

![Final validation results](../output/a1_final/figures/final_f1_seeds.png)

[Report and project materials](../README.md) · [Reproduction and checkpoints](../docs/a1_submission/REPRODUCIBILITY_FINAL.md) · [Data verification](../verification/FINAL_PUBLIC_EVIDENCE_REVIEW.json)
